# Serving & Model Registry


This notebook wraps the models we built in REC:03–REC:06 with three thin layers: a **model registry** handling versioning + stage transitions (MLflow when you're running a server, pickle-on-disk when you aren't), a **FastAPI service** exposing `/v1/recommend` over HTTP, and a **feedback log** that the demo app (REC:09) writes back to. Each of these is small. The interesting part is what they buy us.

Three problems they solve: (1) **online/offline feature parity** — the recommendation made at serving time uses the same `FeatureStore` paths the training notebook called, by construction; (2) **rollback** — when the new model pushes bad metrics, transition the previous version back to Production in one API call; (3) **cost-aware release** — a spent retrain writes a new version to Staging; the demo toggles between Production and Staging without code changes.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")
import os; os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
import time
import numpy as np
import pandas as pd
import torch; torch.manual_seed(0)

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split
from notebooks.recsys.features import FeatureStore
from notebooks.recsys.metrics import Metricator
from notebooks.recsys.models.classic import ALSRecommender
from notebooks.recsys.models.retrieval import TwoTower, TwoTowerConfig, InBatchSoftmaxLoss, TwoTowerTrainer, Retriever
from notebooks.recsys.registry import DiskRegistry, default_registry
from notebooks.recsys.serving import build_app, RecServiceConfig


## The model registry abstraction

Every model starts at stage `none`, gets promoted to `staging` for evaluation, becomes `production` when metrics justify it, and gets `archived` when a newer model takes its place. A registry's job is to remember each version, its associated metrics, and to *guarantee that at most one Production version per model name*. Both our backends (`DiskRegistry`, `MLflowRegistry`) implement this contract.

| stage      | meaning                                                       |
|------------|---------------------------------------------------------------|
| none       | logged but not promoted — default after `log()`              |
| staging    | evaluation/canary; surfaced by `load(stage="staging")`         |
| production | serves the live app; only one version per model              |
| archived   | retired; loaded by explicit version only                      |

Backends abstract this so that the boot path is single-line:

* `default_registry(use_mlflow=False)` → DiskRegistry
* `default_registry(use_mlflow=True)` → MLflowRegistry (requires `MLFLOW_TRACKING_URI`)


In [ ]:
# Use a clean test directory. Set RECSYS_REGISTRY_DIR in your env to /tmp/cleanup or anywhere outside repo.
import shutil, tempfile
test_dir = tempfile.mkdtemp(prefix='rec_registry_')
registry = DiskRegistry(base_dir=test_dir)
print('registry dir:', test_dir)


## Train two competing retriever versions

We'll train two Two-Tower versions — call them v1 and v2 — and decide via offline metrics which one becomes Production. The simlavka here mirrors what a real Canary release looks like: log every training run to the registry, evaluate offline, promote. The registry holds accurate history.


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
train, val = time_split(ds.ratings, val_frac=0.2)

metricator = Metricator(val)
store = FeatureStore(ds)

def train_retriever(embedding_dim, epochs):
    torch.manual_seed(0)
    cfg = TwoTowerConfig(n_users=ds.n_users, n_items=ds.n_items,
                          embedding_dim=embedding_dim, hidden_dim=64,
                          n_negatives=0, epochs=epochs, batch_size=1024)
    m = TwoTower(cfg)
    loss = InBatchSoftmaxLoss(n_items=ds.n_items, n_negatives=0, temperature=0.1)
    TwoTowerTrainer(model=m, loss=loss, cfg=cfg).fit(train, verbose=False)
    return Retriever(model=m, dataset=ds, train=train), cfg

# v1: smaller embeddings, fewer epochs
v1, _ = train_retriever(embedding_dim=16, epochs=1)
v1_m = metricator.evaluate(v1.recommend, k=50)
v1_version = registry.log('twotower', v1, metrics={'recall@50': v1_m['recall@50'], 'ndcg@50': v1_m['ndcg@50']})
# v2: larger, more epochs
v2, _ = train_retriever(embedding_dim=32, epochs=3)
v2_m = metricator.evaluate(v2.recommend, k=50)
v2_version = registry.log('twotower', v2, metrics={'recall@50': v2_m['recall@50'], 'ndcg@50': v2_m['ndcg@50']})
print(f"v1: recall@50 = {v1_m['recall@50']:.4f}")
print(f"v2: recall@50 = {v2_m['recall@50']:.4f}")


In [ ]:
rows = []
for entry in registry.list_models('twotower'):
    rows.append({k: entry.get(k) for k in ['name','version','stage','metrics']})
pd.DataFrame(rows)


Each row is one model *version*. They all sit in `stage=none` until we promote. Below we decide based on `recall@50` and promote the winner.


In [ ]:
if v1_m['recall@50'] >= v2_m['recall@50']:
    registry.transition('twotower', v1_version, 'production')
    print(f"Promoted v{v1_version} to Production")
else:
    registry.transition('twotower', v2_version, 'production')
    print(f"Promoted v{v2_version} to Production")
    # In the unlikely event that v1 has the same metrics, archive v1 explicitly to keep things tidy.
    registry.transition('twotower', v1_version, 'archived')

rows = [{k: e.get(k) for k in ['name','version','stage','metrics']}
        for e in registry.list_models('twotower')]
pd.DataFrame(rows)


Note that exactly one Production version survives — if we had previously promoted v1 we would archive it.


## Wrap the retriever in an HTTP service

Now build the FastAPI service, register our `twotower` Production model under that name, and read recommendations from the network. The same code runs in a notebook (via `TestClient`) and in a real deployment (via `uvicorn notebooks.recsys.serving:app --port 8000`).


In [ ]:
# Bootstrap the service to point at the same registry we populated above.
app = build_app(registry=registry, cfg=RecServiceConfig(default_model='twotower'))
from fastapi.testclient import TestClient
client = TestClient(app)
client.get('/health').json()


In [ ]:
# List registered models via the API
client.get('/v1/models').json()


In [ ]:
# Picks the Production model by default.
uid = int(val['user_id'].iloc[0])
resp = client.get(f'/v1/recommend?user_id={uid}&k=10')
print('HTTP', resp.status_code, 'recs', resp.json())


In [ ]:
# Push a feedback event; this is what the demo app's click handler will call.
client.post('/v1/feedback', json={'user_id': uid, 'item_id': int(resp.json()[0]), 'event': 'click'}).json()


In [ ]:
# Service metrics accumulate across requests so you can spot regressions from the load balancer dashboard.
client.get('/v1/metrics').json()


**Observation.** Every recommend call adds one row to the in-memory metrics. The `avg_latency_ms` field exposes a single p50-style rolling value, useful for the canary-decision-loop: more Production traffic → realistic latency snapshot → re-train decision. In REC:08 we'll wire a counterfactual evaluator to the same `/v1/feedback` log.


::: {.callout-note}
Latency budget: about $100$ ms is the conventional responsiveness budget for the first paint of a rec list. With our FAISS lookup over $\sim$$ 1700$ items Two-Tower pushes that to $\sim$$ 1$ ms plus the overhead of the FastAPI/uvicorn stack (typically $\sim$$ 5$ ms). The wider budget is consumed by the ranker: a DCN forward pass over 200 candidates is $\sim$$ 10$ ms — leaving comfortably $80\sim 90$ ms of slack.
:::


## Promoting v1 back (rollback) is one line

If a canary push goes bad — for instance if `recall@50` was measured on a stale val split but live CTR drops — re-promote the previous version. No retrain.


In [ ]:
# Both versions still in registry. Promote v1 (or v2) regardless of metric; trust sightings on prod.
registry.transition('twotower', v1_version, 'production')
rows = [{k: e.get(k) for k in ['name','version','stage']}
        for e in registry.list_models('twotower')]
pd.DataFrame(rows)


In [ ]:
# Re-hit \(.../recommend): the API now serves v1 again, no service-restart.
client.get(f'/v1/recommend?user_id={uid}&k=5').json()


## MLflow endpoint

If you want the registry backed by MLflow's tracking server you can run one locally:

```bash
# In one shell:
uv run mlflow server --host 0.0.0.0 --port 5000 \
  --backend-store-uri sqlite:///data/mlflow.db \
  --default-artifact-root file://$(pwd)/data/mlruns
# Then in the notebook shell:
export MLFLOW_TRACKING_URI=http://localhost:5000
export RECSYS_USE_MLFLOW=1
```

`default_registry()` reads `RECSYS_USE_MLFLOW` and uses the appropriate backend. The same `build_app(registry=default_registry())` plugs in the mlflow registry for serving.


## Caveats and link forward

- The feature store today is loaded into the service and shared across versions — that works because we use Postgres-style batch reads on ml-100k, but breaks at any reasonable production scale. Swap the in-memory `FeatureStore` with a `RedisFeatureStore` (same interface — see REC:02's swap-in plan) before going multi-process.
- The feedback log is in-memory. Restarting the service drains it. The fix is a database/sqlite-backed `FeedbackStore`; we add a simple version in REC:08.
- We evaluated offline metrics over the time-based val split; live traffic *also* observes users whose val items were held-out under the *first* training policy. **Off-policy evaluation** (REC:08) is what guards against the optimistic-baseline-success problem.

Next: REC:08 — how to evaluate a model on log data that was *generated by another model*.
